In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline

# Load dataset
df = pd.read_csv('titanic_data.csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Features of the data:
- Survived: Survived (0 = No, 1 = Yes)
- Pclass: Which class (1 = Upper class, 2 = Middle class, 3 = Lower class)
- Name: Name of passenger
- Sex: Gender of passenger
- Age: Age of passenger (some NaN)
- SibSp: Number of siblings and spouses of the passenger aboard
- Parch: Number of parents and children of the passenger aboard
- Ticket: Ticket number of passenger
- Fare: Fare paid by passenger
- Cabin: Cabin number of passenger (some NaN)
- Embarked: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southhampton)

Because we wanna know if a passenger has survived or not, we remove the Survived feature from this dataset and store it as its own separate variable 'outcomes'

In [5]:
outcomes = df['Survived']
features_raw = df.drop('Survived', axis = 1)

features_raw.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Preprocessing the data

In [6]:
# One-hot encode features
features = pd.get_dummies(features_raw)

# fill in blanks with zeroes
features = features.fillna(0.0)
features.head(5)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,"Name_Abbing, Mr. Anthony","Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)","Name_Abelson, Mr. Samuel",...,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_C,Embarked_Q,Embarked_S
0,1,3,22.0,1,0,7.2500,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,1,38.0,1,0,71.2833,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,3,26.0,0,0,7.9250,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,1,35.0,1,0,53.1000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,5,3,35.0,0,0,8.0500,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


# Training the model

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features, outcomes, test_size=0.2, random_state=42)

In [11]:
# Import the classifier from sklearn 
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier()

# Testing the model

In [12]:
# Making predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate the accuracy
from sklearn.metrics import accuracy_score
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Training accuracy: {train_accuracy}')
print(f'Test accuracy: {test_accuracy}')

Training accuracy: 1.0
Test accuracy: 0.8100558659217877


# Exercise improving model


We have a high training accuracy and a lower testing accuracy. This means we probably be overfitting.

Use those parameters to improve the testing accuracy:
- max_depth
- min_samples_leaf
- min_samples_split

In [13]:
# Train the model 
model = DecisionTreeClassifier(max_depth=6, min_samples_leaf=6, min_samples_split=10, random_state=42)
model.fit(X_train, y_train)

# Making predictions 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f'Improved trained model accuracy: {train_accuracy}')
print(f'Improved test model accuracy: {test_accuracy}')

Improved trained model accuracy: 0.8707865168539326
Improved test model accuracy: 0.8547486033519553


In [16]:
from sklearn.model_selection import GridSearchCV

model = DecisionTreeClassifier(random_state=42)

tuning_parameters = {'max_depth': range(2, 11),
                     'min_samples_leaf': range(2, 11),
                     'min_samples_split': range(2, 11)}

grid_obj = GridSearchCV(model, tuning_parameters, scoring='f1', verbose=10, n_jobs=2)
grid_fit = grid_obj.fit(X_train, y_train)
model = grid_fit.best_estimator_
#print(f'grid_fit.best_estimator:', grid_fit.best_estimator_)
#print(f'grid_fit.best_score:', grid_fit.best_score_)
#print(f'grid_fit.best_params:', grid_fit.best_params_)

# Making predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculating accuracies 
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print('Training accuracy:', train_accuracy)
print('Test accuracy:', test_accuracy)


Fitting 5 folds for each of 729 candidates, totalling 3645 fits
[CV 5/5; 680/729] END max_depth=10, min_samples_leaf=5, min_samples_split=6;, score=0.686 total time=   0.0s
[CV 1/5; 681/729] START max_depth=10, min_samples_leaf=5, min_samples_split=7...
[CV 1/5; 681/729] END max_depth=10, min_samples_leaf=5, min_samples_split=7;, score=0.695 total time=   0.0s
[CV 2/5; 681/729] START max_depth=10, min_samples_leaf=5, min_samples_split=7...
[CV 2/5; 681/729] END max_depth=10, min_samples_leaf=5, min_samples_split=7;, score=0.705 total time=   0.0s
[CV 3/5; 681/729] START max_depth=10, min_samples_leaf=5, min_samples_split=7...
[CV 3/5; 681/729] END max_depth=10, min_samples_leaf=5, min_samples_split=7;, score=0.720 total time=   0.0s
[CV 4/5; 681/729] START max_depth=10, min_samples_leaf=5, min_samples_split=7...
[CV 4/5; 681/729] END max_depth=10, min_samples_leaf=5, min_samples_split=7;, score=0.679 total time=   0.0s
[CV 3/5; 683/729] START max_depth=10, min_samples_leaf=5, min_sampl